In [1]:
import pandas as pd

out = './mbdump_small/'

# === LOAD ===
history       = pd.read_csv(f'{out}listening_history.tsv', sep='\t')
users         = pd.read_csv(f'{out}users.tsv',             sep='\t')
friends       = pd.read_csv(f'{out}friends.tsv',           sep='\t')
fav_artists   = pd.read_csv(f'{out}fav_artists.tsv',       sep='\t')

history['timestamp'] = pd.to_datetime(history['timestamp'])

# === 1. Implicit score → ALS ===
implicit = (history
    .groupby(['user_id','recording_id'])
    .agg(plays=('recording_id','count'),
         avg_duration=('duration_ms','mean'),
         completion_rate=('completed','mean'))
    .reset_index())

# === 2. Time-of-day profile → playlist model ===
tod_profile = (history
    .groupby(['user_id','recording_id','time_of_day'])
    .size().reset_index(name='play_count'))

# === 3. Sessions → song2vec ===
sessions = (history
    .sort_values('timestamp')
    .groupby(['user_id', pd.Grouper(key='timestamp', freq='30min')])['recording_id']
    .apply(list).reset_index())
sessions.columns = ['user_id', 'session_start', 'recording_sequence']
sessions = sessions[sessions['recording_sequence'].map(len) > 1]  # drop single-song sessions

# === 4. Friend interaction matrix ===
friend_history = (friends
    .merge(history, left_on='friend_id', right_on='user_id', suffixes=('','_friend'))
    .groupby(['user_id','recording_id'])
    .agg(friend_plays=('recording_id','count'))
    .reset_index())

# === 5. Implicit score weighted ===
implicit['implicit_score'] = (
    implicit['plays'] * 1.0 +
    implicit['completion_rate'] * 2.0 +
    (implicit['avg_duration'] / 300000).clip(0, 1) * 1.0
)

# === SAVE ===
feat = f'{out}features/'
import os
os.makedirs(feat, exist_ok=True)

implicit.to_csv(      f'{feat}implicit.tsv',       sep='\t', index=False)
tod_profile.to_csv(   f'{feat}tod_profile.tsv',    sep='\t', index=False)
sessions.to_csv(      f'{feat}sessions.tsv',        sep='\t', index=False)
friend_history.to_csv(f'{feat}friend_history.tsv', sep='\t', index=False)

# === SUMMARY ===
for name, df in [('implicit', implicit), ('tod_profile', tod_profile),
                 ('sessions', sessions), ('friend_history', friend_history)]:
    print(f"{name:25s} → {len(df)} rows")

implicit                  → 13451 rows
tod_profile               → 14327 rows
sessions                  → 2515 rows
friend_history            → 113944 rows


In [ ]:
# pip install implicit gensim mlflow scikit-learn
# hdfs dfs -ls /aulas/francisco_jose_simoes
# hdfs dfs -mkdir /aulas/francisco_jose_simoes/project/data
# hdfs dfs -put /*.tsv /aulas/francisco_jose_simoes/project/data
# docker exec -u root -it ${containerName} bash
# mv file file.tsv (some files no extension)
# need a folder per table
# for f in artist artist_credit artist_credit_name artist_tag fav_artists friends l_recording_recording listening_history listening_history_real notifications recording recording_tag release release_group streaming_events tag track users; do
#   hdfs dfs -mkdir /aulas/francisco_jose_simoes/project/data/$f
#   hdfs dfs -mv /aulas/francisco_jose_simoes/project/data/$f.tsv /aulas/francisco_jose_simoes/project/data/$f/
# done